# Lab 6

**Lab 5 proposal for reference**

**The data that we are going to be working with is the ATLAS LHC simulation 250-500 GeV energy range datasets. Specifically the Higgs signal and multi-jet QCD background for 250-500 GeV in the transverse momentum range.**

**The project that I plan to do will try and address the question: How precise can we get to the Higgs mass and how does narrowing the mass window impact the statistical discovery power?**

**So over the next few labs, I plan to work on multi-parameter optimization where I'll map a 2D/3D parameter space and scan the lower/upper mass window boundaries alongside the d2 substructure cut. Then follow up by performing signal extraction and peak modeling by implementing a curve-fit to fit a background and signal model to the mass distribution. Then work on signal error and test how the significance $\sigma = N_s / \sqrt N_b$ shifts when varying the expected signal/background yields by $\pm 10 \%$**

GOAL FOR TODAY: 

I'm going to be working on the optimization framework.

- Target yields and weighting
- 2D grid search loop 

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [8]:
# Loading in datasets (Higgs signal and QCD background)
higgs_df = pd.read_pickle('../Lab_5/Sample_pt_250_500/higgs_100000_pt_250_500.pkl')
qcd_df = pd.read_pickle('../Lab_5/Sample_pt_250_500/qcd_100000_pt_250_500.pkl')

In [9]:
# Setting up the target physical yields
targ_sig = 100     # Signal
targ_bg = 10000    # Background
# These yield a baseline significance of 1 sigma, serves as a good starting point. 
# I want to modify the numbers later to see if I can run the code on harder/noisier scenarios. 

In [10]:
# Calculating sample weights by dividing our targets by the total number of events in each data frame
weight_sig = targ_sig / len(higgs_df)
weight_bg = targ_bg / len(qcd_df)

# Baseline significance before any cuts have been done. 
baseline_sig = targ_sig / np.sqrt(targ_bg)

print(f'Baseline Significance: {baseline_sig:.2f} sigma')

Baseline Significance: 1.00 sigma


In [11]:
# Grid search

# Scanning the lower and upper bounds of mass. As well as the d2 thresholds.
# Looking at the Mass vs. D2 Higgs signal graph, I set my lower bound between 100 and 120 since the most dense part seems to be at 125. While my upper bound
# is set from 130 to 150. The D2 y-axis was the same with the center at about 1.25 so I took a range from 0.5 to 1.5

mins_mass = np.arange(100, 120, 2) # 100, 102, ..., 118, 120 GeV
maxs_mass = np.arange(130, 150, 2) # 130, 132, ..., 148, 150 GeV
d2_cuts = np.linspace(0.5, 1.5, 50) # Scanning the thresholds from 0.5 and 1.5

new_sig = 0 # Setting baseline for the highest significance found so far
new_config = (0, 0, 0) # Represents min, max, and d2 that gave best_sig

heatmap_results = np.zeros((len(mins_mass), len(d2_cuts))) # Creating a 2D NumPy array filled with zeros. 
print("Running 2D optimization scan...")

Running 2D optimization scan...


In [12]:
for i, min_m in enumerate(mins_mass): # Loops through ech lower mass threshold and gives the index i and actual value min_m.
    for j, d2_cut in enumerate(d2_cuts): # Looping through d2 thresholds for every min_m givign index j and value.
        pass

In [14]:
# Creating empty list to store results
scan_history = []

for min_m in mins_mass:
    for m_max in maxs_mass:
        #
        sig_mass = higgs_df[(higgs_df['mass'] >= min_m) & (higgs_df['mass'] <= m_max)]
        bg_mass = qcd_df[(qcd_df['mass'] >= min_m) & (qcd_df['mass'] <= m_max)]

        for d2_cut in d2_cuts:
            n_s = len(sig_mass[sig_mass['d2'] <= d2_cut])
            n_bg = len(bg_mass[bg_mass['d2'] <= d2_cut])

            N_s = n_s * weight_sig
            N_bg = n_bg * weight_bg

            if N_bg > 0:
                sigma = N_s / np.sqrt(N_bg)
                scan_history.append({'m_min': min_m, 'm_max': m_max, 'd2': d2_cut, 'signifance': sigma})

                if sigma > new_sig:
                    new_sig = sigma
                    new_config = (min_m, m_max, d2_cut)


print(f'Max Significance: {new_sig:.3f} sigma')
print(f'Best Mass Window: [{new_config[0]} GeV, {new_config[1]} GeV]')
print(f'Best Max d2 Cut: <= {new_config[2]:.3f}')

Max Significance: 41.647 sigma
Best Mass Window: [118 GeV, 132 GeV]
Best Max d2 Cut: <= 0.643


The max significance value seems weird. It seems to be too high, not sure why. I'll continue to investigate why this is in Lab 7. Aiming to bring the Max significance down. 